# 01_eda

データの中身を見て，特徴をつかむ．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error

from bike_sharing.data import load_data
from bike_sharing.visualize import set_style

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
set_style()

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

In [ ]:
train.head()

## 1. データの形

In [ ]:
train.info()

In [ ]:
print("test の欠損:", test.isnull().sum().sum())

In [ ]:
train.describe()

In [ ]:
# datetime から年・月・日・時間・曜日を取り出す
for df in [train, test]:
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour
    df["dow"] = df["datetime"].dt.dayofweek  # 月曜が 0，日曜が 6

# 評価が RMSLE なので log(1 + count) も作っておく
train["log_count"] = np.log1p(train["count"])

In [ ]:
# train と test で日がどう分かれているか
print("train:", train["day"].unique())
print("test:", test["day"].unique())

In [ ]:
# 1 日ごとの合計．test の日は train に無いので線が切れる
daily = train.set_index("datetime")["count"].resample("D").sum(min_count=1)

plt.figure(figsize=(12, 4))
plt.plot(daily)
plt.title("1 日ごとの貸出台数")
plt.ylabel("台数")
plt.show()

## 2. count の分布と RMSLE

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(train["count"], bins=50)
ax[0].set_title("count")
ax[1].hist(train["log_count"], bins=50)
ax[1].set_title("log(1 + count)")
plt.show()

print("歪度 count:", train["count"].skew())
print("歪度 log(1 + count):", train["log_count"].skew())

In [ ]:
# 全部同じ値で予測したときのスコア．これより悪かったら意味がない
# log(1 + count) の平均を戻した値を使う（RMSLE ならこれが一番よくなるはず）
pred = np.expm1(train["log_count"].mean())
print("予測値:", pred)
print("RMSLE:", root_mean_squared_log_error(train["count"], np.full(len(train), pred)))

In [ ]:
# RMSLE は比で効くので，台数が少ないときのずれが大きくなる
print("1 台を 5 台と予測:", np.log1p(5) - np.log1p(1))
print("500 台を 600 台と予測:", np.log1p(600) - np.log1p(500))

## 3. 時間帯ごとの台数（casual / registered）

In [ ]:
# holiday が 1 の日が何曜日か
pd.crosstab(train["dow"], train["holiday"])

In [ ]:
# holiday の日は全部平日だったので，workingday = 0 は土日か祝日になる
train["day_type"] = "土日"
train.loc[train["workingday"] == 1, "day_type"] = "勤務日"
train.loc[train["holiday"] == 1, "day_type"] = "祝日"

# それぞれ何日あるか
train.drop_duplicates(subset=["year", "month", "day"])["day_type"].value_counts()

In [ ]:
reg = train.pivot_table(index="hour", columns="day_type", values="registered", aggfunc="mean")
cas = train.pivot_table(index="hour", columns="day_type", values="casual", aggfunc="mean")

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for t in ["勤務日", "土日", "祝日"]:
    ax[0].plot(reg.index, reg[t], marker=".", label=t)
    ax[1].plot(cas.index, cas[t], marker=".", label=t)
ax[0].set_title("registered")
ax[1].set_title("casual")
ax[0].set_xlabel("時間")
ax[1].set_xlabel("時間")
ax[0].legend()
ax[1].legend()
plt.show()

In [ ]:
print("casual の割合:", train["casual"].sum() / train["count"].sum())

## 4. 年・月ごとの台数

In [ ]:
monthly = train.pivot_table(index="month", columns="year", values="count", aggfunc="mean")

plt.figure(figsize=(7, 4))
plt.plot(monthly.index, monthly[2011], marker="o", label="2011")
plt.plot(monthly.index, monthly[2012], marker="o", label="2012")
plt.title("月ごとの平均台数")
plt.xlabel("月")
plt.legend()
plt.show()

In [ ]:
# 2012 年が 2011 年の何倍か
print((monthly[2012] / monthly[2011]).round(2))

In [ ]:
yearly = train.groupby("year")[["casual", "registered", "count"]].mean()
yearly.loc[2012] / yearly.loc[2011]

## 5. 天気・気温

In [ ]:
# 気温を 2 度ごとに区切って，log(1 + count) の平均を見る（行が少ない区間は除く）
tmp = train.groupby(train["temp"] // 2 * 2)["log_count"].agg(["mean", "count"])
tmp = tmp[tmp["count"] >= 30]

plt.figure(figsize=(6, 4))
plt.plot(tmp.index, tmp["mean"], marker=".")
plt.title("temp")
plt.xlabel("temp")
plt.ylabel("log(1 + count) の平均")
plt.show()

In [ ]:
# 湿度は 5 ごと
tmp = train.groupby(train["humidity"] // 5 * 5)["log_count"].agg(["mean", "count"])
tmp = tmp[tmp["count"] >= 30]

plt.figure(figsize=(6, 4))
plt.plot(tmp.index, tmp["mean"], marker=".")
plt.title("humidity")
plt.xlabel("humidity")
plt.ylabel("log(1 + count) の平均")
plt.show()

In [ ]:
# 風速は 3 ごと
tmp = train.groupby(train["windspeed"] // 3 * 3)["log_count"].agg(["mean", "count"])
tmp = tmp[tmp["count"] >= 30]

plt.figure(figsize=(6, 4))
plt.plot(tmp.index, tmp["mean"], marker=".")
plt.title("windspeed")
plt.xlabel("windspeed")
plt.ylabel("log(1 + count) の平均")
plt.show()

In [ ]:
print(train["weather"].value_counts())
print(test["weather"].value_counts())
train.groupby("weather")["log_count"].mean()

## 6. 相関

### 6.1 特徴量どうし

In [ ]:
cols = ["hour", "dow", "month", "year", "season", "holiday", "workingday", "weather",
        "temp", "atemp", "humidity", "windspeed"]
corr = train[cols].corr()
corr.round(2)

In [ ]:
# 相関が高かったところ
print("temp - atemp:", corr.loc["temp", "atemp"])
print("month - season:", corr.loc["month", "season"])
print("dow - workingday:", corr.loc["dow", "workingday"])

### 6.2 log(1 + count) との関係

In [ ]:
train[cols + ["log_count"]].corr()["log_count"].round(3)

In [ ]:
train[cols + ["log_count"]].corr(method="spearman")["log_count"].round(3)

相関係数は直線の関係しか見ないので，相関比 η² も出してみる．グループごとの平均でどれくらい説明できるかの割合で，η² = 1 - (グループ内の平方和) / (全体の平方和)．

In [ ]:
total = ((train["log_count"] - train["log_count"].mean()) ** 2).sum()

for col in ["hour", "dow", "month", "year", "season", "holiday", "workingday", "weather", "windspeed"]:
    group_mean = train.groupby(col)["log_count"].transform("mean")
    within = ((train["log_count"] - group_mean) ** 2).sum()
    r = train[col].corr(train["log_count"])
    print(col, " 相関係数の2乗:", round(r**2, 3), " η²:", round(1 - within / total, 3))

In [ ]:
# workingday 単体だとほぼ 0 だけど，hour と組み合わせるとどうなるか
for keys in [["hour"], ["workingday"], ["hour", "workingday"], ["hour", "workingday", "year"]]:
    group_mean = train.groupby(keys)["log_count"].transform("mean")
    within = ((train["log_count"] - group_mean) ** 2).sum()
    print(keys, round(1 - within / total, 3))

### 6.3 時間の影響を除いた相関

humidity などは時間と一緒に動いている（夜は湿度が高い）ので，時間 × workingday ごとの平均を引いた残りとの相関も見る．

In [ ]:
train["resid"] = train["log_count"] - train.groupby(["hour", "workingday"])["log_count"].transform("mean")

for col in ["temp", "atemp", "humidity", "windspeed", "weather"]:
    print(col, round(train[col].corr(train["log_count"]), 3), "->", round(train[col].corr(train["resid"]), 3))

## 7. おかしな値

In [ ]:
# windspeed の値を小さい順に見る
print(np.sort(train["windspeed"].unique())[:10])
print("0 の割合 train:", (train["windspeed"] == 0).mean())
print("0 の割合 test:", (test["windspeed"] == 0).mean())

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(train["windspeed"], bins=60)
plt.title("windspeed")
plt.show()

In [ ]:
# 0 がどの時間に多いか
zero_rate = train.groupby("hour")["windspeed"].apply(lambda x: (x == 0).mean())

plt.figure(figsize=(6, 4))
plt.plot(zero_rate.index, zero_rate.values, marker=".")
plt.title("windspeed = 0 の割合")
plt.xlabel("時間")
plt.show()

In [ ]:
# humidity が 0 の行
train[train["humidity"] == 0]["datetime"].dt.date.value_counts()

In [ ]:
# 前後の日と比べる
train[(train["year"] == 2011) & (train["month"] == 3) & (train["day"].between(9, 11))].groupby("day")["humidity"].mean()

In [ ]:
# atemp と temp の差
train["diff"] = train["atemp"] - train["temp"]
train["diff"].describe()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(train["temp"], train["atemp"], s=3)
plt.xlabel("temp")
plt.ylabel("atemp")
plt.show()

In [ ]:
# 1 本だけ横に外れている線がある．差が -10 より小さい行を見る
odd = train[train["diff"] < -10]
print(len(odd), "行")
print(odd["datetime"].dt.date.unique())
print("atemp:", odd["atemp"].unique())
print("test の差の最小:", (test["atemp"] - test["temp"]).min())

In [ ]:
# weather = 4 の行
print(train[train["weather"] == 4][["datetime", "temp", "humidity", "count"]])
print(test[test["weather"] == 4][["datetime", "temp", "humidity"]])

## 8. 抜けている時間

In [ ]:
# 2 年分の全部の時間のうち，train にも test にも無い時間
all_hours = pd.date_range("2011-01-01 00:00", "2012-12-31 23:00", freq="h")
both = pd.concat([train["datetime"], test["datetime"]])
missing = pd.DataFrame({"datetime": all_hours[~all_hours.isin(both)]})
missing["date"] = missing["datetime"].dt.date
missing["day"] = missing["datetime"].dt.day
missing["hour"] = missing["datetime"].dt.hour

print("抜けている時間:", len(missing))
print("train の期間:", (missing["day"] <= 19).sum())
print("test の期間:", (missing["day"] >= 20).sum())

In [ ]:
# 抜けが多い日
missing["date"].value_counts().head(10)

In [ ]:
# 6 時間以上抜けている日を除いて，残りが何時に多いか
cnt = missing["date"].value_counts()
big_days = cnt[cnt >= 6].index
small = missing[~missing["date"].isin(big_days)]

plt.figure(figsize=(6, 4))
plt.bar(small["hour"].value_counts().index, small["hour"].value_counts().values)
plt.title("抜けている時間（まとまった日以外）")
plt.xlabel("時間")
plt.show()

In [ ]:
# count が少ない時間も同じあたりか
print("count の最小:", train["count"].min())
train[train["count"] <= 2]["hour"].value_counts().sort_index()

## 9. 祝日

In [ ]:
print("train:", list(train[train["holiday"] == 1]["datetime"].dt.strftime("%Y-%m-%d %a").unique()))
print("test:", list(test[test["holiday"] == 1]["datetime"].dt.strftime("%Y-%m-%d %a").unique()))